In [ ]:
import re
import json
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def parse_triple(triple):
    pattern = re.compile(r"\[entity1, ([^\]]+)\]: (.+?), \[relation, ([^\]]+)\]: (.+?), \[entity2, ([^\]]+)\]: (.+)")
    match = pattern.search(triple["triples"])
    if match:
        entity1_id, entity1, _, _, entity2_id, entity2 = match.groups()
        return (entity1_id, entity1.strip()), (entity2_id, entity2.strip())
    else:
        raise ValueError("Triple format is incorrect")

def get_wikipedia_intro_by_paragraph(entity_label):
    url = f"https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "exintro": True,
        "titles": entity_label,
        "format": "json",
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        pages = data.get("query", {}).get("pages", {})
        for page_id, page_data in pages.items():
            if page_id != "-1":
                html_content = page_data.get("extract", "No description found")
                soup = BeautifulSoup(html_content, "html.parser")
                paragraphs = soup.find_all('p')
                non_empty_paragraphs = [p.get_text().strip() for p in paragraphs if p.get_text().strip()]
                if len(non_empty_paragraphs) >= 2:
                    return "\n\n".join(non_empty_paragraphs[:2])
                else:
                    return "\n\n".join(non_empty_paragraphs)
    return "No description found"

def extract_entities_and_match_text(jsonl_file, output_file):
    extracted_entities = {}

    def fetch_and_store(entity_id, entity_name):
        if entity_id not in extracted_entities:
            entity_text = get_wikipedia_intro_by_paragraph(entity_name)
            extracted_entities[entity_id] = f"[{entity_name}, {entity_id}]: {entity_text}"

    with open(jsonl_file, 'r', encoding='utf-8') as file:
        triples = [json.loads(line) for line in file]

    with ThreadPoolExecutor(max_workers=100) as executor:
        futures = []
        for data in triples:
            try:
                (entity1_id, entity1), (entity2_id, entity2) = parse_triple(data)
                futures.append(executor.submit(fetch_and_store, entity1_id, entity1))
                futures.append(executor.submit(fetch_and_store, entity2_id, entity2))
            except ValueError as e:
                print(f"Skipping line due to error: {e}")

        for _ in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
            pass

    with open(output_file, 'w', encoding='utf-8') as outfile:
        for entity_text in extracted_entities.values():
            json.dump({"paragraph": entity_text}, outfile, ensure_ascii=False)
            outfile.write('\n')

jsonl_file = "/hkfs/home/project/hk-project-p00201316/st_st176945/testin/10_reformed.jsonl"
output_file = "/hkfs/home/project/hk-project-p00201316/st_st176945/testout/output.jsonl"

extract_entities_and_match_text(jsonl_file, output_file)